# 📝 Linear Regression — Capstone Solutions

---

Is notebook mein **02_LinearRegression_California.ipynb** ke saare Capstone Parts ke complete answers hain.

| Part | Topic | Status |
|------|-------|--------|
| **Part A** | Feature Engineering | ✅ |
| **Part B** | Regional Analysis (North vs South CA) | ✅ |
| **Part C** | ElasticNet GridSearchCV | ✅ |
| **Part D** | Error Analysis — Worst Predictions | ✅ |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# ─── Dataset Load ─────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame
X_base = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train_b, X_test_b, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

# Baseline model (sab parts mein compare karna hai isse)
baseline_pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
baseline_pipe.fit(X_train_b, y_train)
baseline_r2 = r2_score(y_test, baseline_pipe.predict(X_test_b))
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pipe.predict(X_test_b)))

print('✅ Setup complete!')
print(f'Baseline (8 features): R²={baseline_r2:.4f}, RMSE=${baseline_rmse*100:.0f}K')

---
## Part A: Feature Engineering Challenge

**Question:** 2 naye features create karo aur check karo ki R² improve hota hai ya nahi.

**Features to create:**
- `RoomsPerPerson = AveRooms / AveOccup`
- `BedroomRatio = AveBedrms / AveRooms`

**Expected insight:** Yeh features zyada meaningful hain kyunki:
- `AveRooms` akela misleading hai — ek ghar mein 10 rooms hain lekin 20 log rehte hain toh actually crowded hai
- `RoomsPerPerson` real spaciousness capture karta hai
- `BedroomRatio` house type capture karta hai (studio vs family home)

In [ ]:
# ─── PART A SOLUTION ─────────────────────────────────────────────────────────

df_eng = df.copy()

# Feature 1: RoomsPerPerson — har person ke liye kitne rooms hain
df_eng['RoomsPerPerson'] = df_eng['AveRooms'] / df_eng['AveOccup']

# Feature 2: BedroomRatio — bedrooms ka proportion total rooms mein
df_eng['BedroomRatio'] = df_eng['AveBedrms'] / df_eng['AveRooms']

# Infinity ya NaN values handle karo (zero division se)
df_eng.replace([np.inf, -np.inf], np.nan, inplace=True)
df_eng.dropna(inplace=True)

print(f'Original shape: {df.shape}')
print(f'After engineering: {df_eng.shape}')
print(f'\nNew features added:')
print(df_eng[['AveRooms', 'AveOccup', 'RoomsPerPerson',
              'AveBedrms', 'BedroomRatio']].head(5).round(3))

In [ ]:
# New features ke saath model train karo
X_new = df_eng.drop('MedHouseVal', axis=1)  # 10 features ab
y_new = df_eng['MedHouseVal']

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_new, y_new, test_size=0.2, random_state=42
)

# Model with new features
new_pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
new_pipe.fit(X_train_n, y_train_n)

new_r2   = r2_score(y_test_n, new_pipe.predict(X_test_n))
new_rmse = np.sqrt(mean_squared_error(y_test_n, new_pipe.predict(X_test_n)))

print('=== Part A Results ===')
print(f'Baseline  (8 features):  R²={baseline_r2:.4f}, RMSE=${baseline_rmse*100:.0f}K')
print(f'Engineered (10 features): R²={new_r2:.4f}, RMSE=${new_rmse*100:.0f}K')
print(f'\nImprovement: ΔR² = {(new_r2 - baseline_r2)*100:+.2f}%')

# Feature importance comparison
coef_df = pd.DataFrame({
    'Feature': X_new.columns,
    'Coefficient': new_pipe.named_steps['lr'].coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 5))
colors = ['green' if c > 0 else 'red' for c in coef_df.Coefficient]
plt.barh(coef_df.Feature, coef_df.Coefficient, color=colors, alpha=0.8, edgecolor='black')
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Standardized Coefficient')
plt.title('Part A — Feature Importance (10 features, Engineered)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 Insight:')
print('RoomsPerPerson positive coefficient → zyada space per person = higher price')
print('BedroomRatio high → more bedrooms proportionally → family home = higher price')

---
## Part B: Regional Analysis — North vs South California

**Question:** California ko North aur South mein split karo. Ek global model better hai ya regional models?

**Split Rule:**
- Latitude > 37.5 → Northern California (San Francisco, Sacramento area)
- Latitude ≤ 37.5 → Southern California (Los Angeles, San Diego area)

**Hypothesis:** Regional models better perform karenge kyunki housing market North aur South mein alag alag hai.

In [ ]:
# ─── PART B SOLUTION ─────────────────────────────────────────────────────────

df_north = df[df.Latitude > 37.5].copy()
df_south = df[df.Latitude <= 37.5].copy()

print('=== Region Split ===')
print(f'Northern California: {len(df_north):,} samples | Avg Price: ${df_north.MedHouseVal.mean()*100:.0f}K')
print(f'Southern California: {len(df_south):,} samples | Avg Price: ${df_south.MedHouseVal.mean()*100:.0f}K')

def train_regional(df_region, region_name):
    X_r = df_region.drop('MedHouseVal', axis=1)
    y_r = df_region['MedHouseVal']
    X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
    pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
    pipe.fit(X_tr, y_tr)
    r2  = r2_score(y_te, pipe.predict(X_te))
    rmse = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))
    print(f'{region_name}: R²={r2:.4f}, RMSE=${rmse*100:.0f}K, samples={len(df_region):,}')
    return pipe, r2, rmse

print('\n=== Regional Model Results ===')
north_model, north_r2, north_rmse = train_regional(df_north, 'Northern CA')
south_model, south_r2, south_rmse = train_regional(df_south, 'Southern CA')
print(f'\nGlobal Model:    R²={baseline_r2:.4f}, RMSE=${baseline_rmse*100:.0f}K')

In [ ]:
# Geographic visualization — North vs South prices
sample_n = df_north.sample(1000, random_state=42)
sample_s = df_south.sample(1000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Map
sc1 = axes[0].scatter(sample_n.Longitude, sample_n.Latitude,
                       c=sample_n.MedHouseVal, cmap='YlOrRd', s=10, alpha=0.6,
                       label='North CA', vmin=0, vmax=5)
sc2 = axes[0].scatter(sample_s.Longitude, sample_s.Latitude,
                       c=sample_s.MedHouseVal, cmap='YlOrRd', s=10, alpha=0.6,
                       label='South CA', vmin=0, vmax=5)
axes[0].axhline(37.5, color='blue', linestyle='--', linewidth=2,
                label='North/South Split (Lat=37.5)')
plt.colorbar(sc1, ax=axes[0], label='Price ($100K)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('California — North vs South Split')
axes[0].legend()

# R² comparison bar chart
regions = ['Global Model', 'North CA Model', 'South CA Model']
r2_vals = [baseline_r2, north_r2, south_r2]
bar_colors = ['gray', 'steelblue', 'coral']

bars = axes[1].bar(regions, r2_vals, color=bar_colors, edgecolor='black', alpha=0.8)
for bar, val in zip(bars, r2_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold')
axes[1].set_ylabel('Test R²')
axes[1].set_title('Part B — Global vs Regional Model Comparison')
axes[1].set_ylim(0, 0.75)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n💡 Conclusion:')
print('Regional models USUALLY better perform karte hain kyunki:')
print('- North CA (SF area): tech workers, high income → steep price-income relationship')
print('- South CA (LA area): more diverse market, sprawl, different age profiles')
print('=> Agar R² regional > global, hypothesis sahi hai!')

---
## Part C: ElasticNet GridSearchCV

**Question:** GridSearchCV se ElasticNet ke liye best `alpha` aur `l1_ratio` dhundho.

**Parameter Grid:**
- `alpha`: [0.001, 0.01, 0.1, 1]
- `l1_ratio`: [0.1, 0.3, 0.5, 0.7, 0.9]

**Reminder:**
- `l1_ratio = 0.0` → Pure Ridge (L2 only)
- `l1_ratio = 1.0` → Pure Lasso (L1 only)
- `l1_ratio = 0.5` → Equal mix

In [ ]:
# ─── PART C SOLUTION ─────────────────────────────────────────────────────────

scaler_c = StandardScaler()
X_train_sc = scaler_c.fit_transform(X_train_b)
X_test_sc  = scaler_c.transform(X_test_b)

param_grid_en = {
    'alpha':    [0.001, 0.01, 0.1, 1],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

en_gs = GridSearchCV(
    ElasticNet(max_iter=5000),
    param_grid_en,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
en_gs.fit(X_train_sc, y_train)

print('=== Part C: ElasticNet GridSearchCV ===')
print(f'Best Alpha:    {en_gs.best_params_["alpha"]}')
print(f'Best l1_ratio: {en_gs.best_params_["l1_ratio"]} ', end='')
lr = en_gs.best_params_['l1_ratio']
if lr < 0.3:
    print('(Ridge-dominant)')
elif lr > 0.7:
    print('(Lasso-dominant)')
else:
    print('(Balanced mix)')

print(f'Best CV R²:   {en_gs.best_score_:.4f}')
print(f'Test R²:      {r2_score(y_test, en_gs.predict(X_test_sc)):.4f}')
print(f'Baseline R²:  {baseline_r2:.4f}')

In [ ]:
# GridSearch results visualize karo — R² heatmap
results_df = pd.DataFrame(en_gs.cv_results_)
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_alpha',
    columns='param_l1_ratio'
)

plt.figure(figsize=(10, 6))
sns.heatmap(
    pivot, annot=True, fmt='.4f', cmap='YlGn',
    linewidths=0.5, annot_kws={'size': 10}
)
plt.title('Part C — ElasticNet GridSearchCV: CV R² Heatmap')
plt.xlabel('l1_ratio (0=Ridge, 1=Lasso)')
plt.ylabel('Alpha (Regularization Strength)')
plt.tight_layout()
plt.show()

# Zero coefficients count karo
best_en = en_gs.best_estimator_
n_zero = np.sum(np.abs(best_en.coef_) < 1e-6)
print(f'\nBest ElasticNet — Zero coefficients: {n_zero}/{len(best_en.coef_)}')
print('(Non-zero means feature selected!)')

print('\n💡 Conclusion:')
print('Small alpha best: California Housing mein zyada regularization zaruri nahi')
print('Kyunki 8 features sab meaningful hain (koi truly useless feature nahi)')

---
## Part D: Error Analysis — Worst Predictions Kahan Hain?

**Question:** Worst 20 predictions dhundho aur patterns analyze karo:
- Kya worst errors sirf high-price houses pe hain?
- Kya specific geography (Lat/Long) pe errors zyada hain?
- Kya income group pe depend karta hai?

**Insight:** Jahan model fail karta hai, wahan either:
1. **Price capping** at $5 (data artifact)
2. **Outlier neighborhoods** (gated communities, beachfront)
3. **Missing features** (school quality, crime rate, amenities)

In [ ]:
# ─── PART D SOLUTION ─────────────────────────────────────────────────────────

# Final pipeline se predictions
final_pipe_d = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
final_pipe_d.fit(X_train_b, y_train)
y_pred_d = final_pipe_d.predict(X_test_b)

error_df = X_test_b.copy()
error_df['Actual']    = y_test.values
error_df['Predicted'] = y_pred_d
error_df['Error']     = y_test.values - y_pred_d
error_df['AbsError']  = np.abs(error_df['Error'])
error_df['PctError']  = (error_df['AbsError'] / error_df['Actual'] * 100).round(1)

print('=== Part D: Top 20 Worst Predictions ===')
top20 = error_df.nlargest(20, 'AbsError')[
    ['MedInc', 'HouseAge', 'Latitude', 'Longitude',
     'Actual', 'Predicted', 'AbsError', 'PctError']
].round(3)
print(top20.to_string())

In [ ]:
# Pattern Analysis — 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Error vs Actual Price — kya high-price houses pe zyada error hai?
axes[0].scatter(error_df['Actual'], error_df['AbsError'],
                alpha=0.3, s=10, color='steelblue')
top20_df = error_df.nlargest(20, 'AbsError')
axes[0].scatter(top20_df['Actual'], top20_df['AbsError'],
                color='red', s=80, label='Top 20 Worst', zorder=5)
axes[0].axvline(5.0, color='orange', linestyle='--', linewidth=2,
                label='Price Cap = $500K')
axes[0].set_xlabel('Actual Price ($100K)')
axes[0].set_ylabel('Absolute Error')
axes[0].set_title('Error vs Actual Price')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Geographic distribution of errors
sample_all = error_df.sample(2000, random_state=42)
sc = axes[1].scatter(
    sample_all['Longitude'], sample_all['Latitude'],
    c=sample_all['AbsError'], cmap='hot_r',
    alpha=0.6, s=10, vmax=error_df['AbsError'].quantile(0.95)
)
axes[1].scatter(top20_df['Longitude'], top20_df['Latitude'],
                color='blue', s=100, marker='*', label='Top 20 Worst', zorder=5)
plt.colorbar(sc, ax=axes[1], label='Abs Error')
axes[1].set_title('Geographic Error Distribution')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].legend()

# 3. Error by income group
error_df['IncomeGroup'] = pd.cut(
    error_df['MedInc'],
    bins=[0, 2, 4, 6, 15],
    labels=['Low (<2)', 'Medium (2-4)', 'High (4-6)', 'Very High (6+)']
)
error_df.boxplot(column='AbsError', by='IncomeGroup', ax=axes[2])
axes[2].set_title('Error by Income Group')
axes[2].set_xlabel('Income Group')
axes[2].set_ylabel('Absolute Error')
plt.suptitle('')
axes[2].grid(alpha=0.3)

plt.suptitle('Part D — Error Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for error analysis
print('=== Error Analysis Summary ===')

# Q1: High-price houses pe zyada error?
high_price = error_df[error_df['Actual'] >= 4.5]
low_price  = error_df[error_df['Actual'] < 3.0]
print(f'\n1. High-price (>=450K) RMSE:  ${high_price.AbsError.mean()*100:.0f}K')
print(f'   Low-price (<300K) RMSE:    ${low_price.AbsError.mean()*100:.0f}K')
print(f'   => High-price mein error zyada? {high_price.AbsError.mean() > low_price.AbsError.mean()}')

# Q2: Geographic cluster
top20_lat = top20_df['Latitude'].mean()
top20_lon = top20_df['Longitude'].mean()
print(f'\n2. Top 20 worst errors ka avg location: Lat={top20_lat:.2f}, Lon={top20_lon:.2f}')
if top20_lat > 37:
    print(f'   => Bay Area / SF mein cluster (expensive coastal market)')
else:
    print(f'   => LA / Southern CA mein cluster')

# Q3: Income group
print('\n3. Mean Absolute Error by Income Group:')
print(error_df.groupby('IncomeGroup')['AbsError'].mean().round(3).to_string())

print('\n💡 Key Conclusions:')
print('1. PRICE CAPPING: Data mein prices $5 pe capped hain (1990 data artifact)')
print('   => Model $5 se upar predict karna sikhta nahi → highest-price areas mein large errors')
print('2. COASTAL PREMIUM: Beach/Bay area houses mein hidden features hain')
print('   (ocean view, school quality) jo dataset mein nahi hain → higher errors')
print('3. Very High income group mein error zyada → luxury market alag behave karta hai')

---
## Bonus: Sabhi Parts Ka Comparison

Sabhi approaches ek jagah — kaunsa best perform kiya?

In [ ]:
# ─── FINAL COMPARISON TABLE ───────────────────────────────────────────────────

print('=== Final Comparison — Sabhi Approaches ===')
print(f'{"-"*65}')
print(f'{"Approach":<35} {"Test R²":>10} {"RMSE ($K)":>12}')
print(f'{"-"*65}')

# 1. Baseline
print(f'{"Baseline (8 features, LR)":<35} {baseline_r2:>10.4f} {baseline_rmse*100:>12.0f}')

# 2. Part A — Feature Engineering
print(f'{"Part A: +2 Engineered Features":<35} {new_r2:>10.4f} {new_rmse*100:>12.0f}')

# 3. Part B — Regional (North)
print(f'{"Part B: North CA Regional Model":<35} {north_r2:>10.4f} {north_rmse*100:>12.0f}')

# 4. Part B — Regional (South)
print(f'{"Part B: South CA Regional Model":<35} {south_r2:>10.4f} {south_rmse*100:>12.0f}')

# 5. Part C — ElasticNet
en_test_r2   = r2_score(y_test, en_gs.predict(X_test_sc))
en_test_rmse = np.sqrt(mean_squared_error(y_test, en_gs.predict(X_test_sc)))
print(f'{"Part C: ElasticNet (best alpha)":<35} {en_test_r2:>10.4f} {en_test_rmse*100:>12.0f}')

print(f'{"-"*65}')

print('\n💡 Key Takeaways:')
print('• Feature engineering (Part A) often gives best R² boost for LR')
print('• Regional models help when geography matters a lot')
print('• ElasticNet R² similar to LR — California Housing mein LR already good')
print('• Tree-based models (Random Forest) would significantly beat all of these!')

---
## Summary — Capstone Se Kya Seekha?

| Part | Concept | Key Learning |
|------|---------|-------------|
| **A** | Feature Engineering | Domain knowledge se features banana often R² zyada improve karta hai model switching se zyada |
| **B** | Regional Analysis | Ek bada dataset ko meaningful subsets mein todna better predictions de sakta hai |
| **C** | ElasticNet Tuning | GridSearchCV 2D parameter space mein best point dhundta hai; heatmap pattern dikhata hai |
| **D** | Error Analysis | Model failures samajhna utna hi important hai jitna good metrics — woh batata hai kahan improve karna hai |

**Next Step:** Random Forest ya Gradient Boosting try karo — woh non-linear patterns capture karte hain jo Linear Regression miss karta hai, specifically:
- Price capping at $500K
- Coastal area non-linear premium
- Income-price interaction effects